In [ ]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from torch.utils.data import Dataset, DataLoader
import pandas as pd

## Data

In [4]:
avisos = pd.read_csv("trainig_data.csv", index_col=0)

## Model BERTin (BETO)

In [ ]:
model_name = "distilbert-base-uncased"

# Downloads the model and the tokenizer
model = AutoModelForSequenceClassification.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

Define a custom PyTorch Dataset class to handle data:

In [37]:
class AvisosDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

Encode the Data: Split the DataFrame into training and validation sets, then encode the text data using the tokenizer:

In [38]:
# Split the dataset
train_texts, val_texts, train_labels, val_labels = train_test_split(
    avisos['aviso'].tolist(), 
    avisos['classification'].astype('category').cat.codes.tolist(), 
    test_size=0.2  # Share of test part
)

# Encode the data
train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=128) # 128 to make it faster due to large ads, benchmark 512
val_encodings = tokenizer(val_texts, truncation=True, padding=True, max_length=128)

# Create the dataset objects
train_dataset = AvisosDataset(train_encodings, train_labels)
val_dataset = AvisosDataset(val_encodings, val_labels)

Set Up Training Arguments

In [ ]:
training_args = TrainingArguments(
    output_dir='./results',          # output directory
    num_train_epochs=2,              # number of training epochs
    per_device_train_batch_size=4,   # batch size for training
    per_device_eval_batch_size=8,    # batch size for evaluation
    warmup_steps=500,                # number of warmup steps for learning rate scheduler
    weight_decay=0.01,               # strength of weight decay
    logging_dir='./logs',            # directory for storing logs
    logging_steps=10,
    evaluation_strategy="steps",     # evaluation strategy
    gradient_accumulation_steps=4,   # Simulate a larger batch size for faster training
    fp16=True,                       # Enable mixed precision training to speed up training
    # Early stopping for speeding up
    load_best_model_at_end=True,     # Save the best model
    save_total_limit=1,              # Limit the number of saved models
    eval_steps=500,                  # Evaluate every 500 steps
    metric_for_best_model="accuracy",
    greater_is_better=True,
    save_strategy="steps",
    save_steps=500,
)

To evaluate the model we use a custom function that reports metrics
- **Accuracy:** Proportion of correct predictions over total predictions.
- **Precision:** Proportion of true positives out of all positive predictions.
- **Recall:** Proportion of true positives out of all actual positives.
- **F1-Score:** Harmonic mean of precision and recall, balancing both metrics.

In [48]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(axis=-1)
    
    accuracy = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='binary')
    
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
    }

Use the Trainer class from Hugging Faces Transformers to train the model:

In [ ]:
trainer = Trainer(
    model=model,                         # the instantiated 🤗 Transformers model to be trained
    args=training_args,                  # training arguments, defined above
    train_dataset=train_dataset,         # training dataset
    eval_dataset=val_dataset,            # evaluation dataset
    compute_metrics=compute_metrics      # metrics for overall evaluation (custom)
)

trainer.train()

Evaluate and Save the Model

In [ ]:
# Evaluate the model
eval_results = trainer.evaluate()

# Save the model
model.save_pretrained('./distilbert-avisos-classification2')
tokenizer.save_pretrained('./distilbert-avisos-classification2')

In [ ]:
eval_results